In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

In [2]:
feature_dir  = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/feature_tensors")
cat_dir      = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/homogenised_catalogs")
temporal_dir = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/temporal_features")
target_dir   = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/targets")

In [3]:
# Patch definitions
PATCHES = {
    "Kanto_Japan":      dict(minlat=34.5,  maxlat=37.2,  minlon=138.5, maxlon=141.5),
    "Tohoku_Japan":     dict(minlat=37.5,  maxlat=40.5,  minlon=140.5, maxlon=143.5),
    "Central_Chile":    dict(minlat=-36.5, maxlat=-33.5, minlon=-72.5, maxlon=-69.5),
    "Central_Turkey":   dict(minlat=36.5,  maxlat=39.0,  minlon=35.5,  maxlon=39.0),
    "Central_Nepal":    dict(minlat=27.0,  maxlat=29.7,  minlon=83.5,  maxlon=86.5),
    "North_Island_NZ":  dict(minlat=-40.5, maxlat=-37.5, minlon=174.5, maxlon=178.0),
    "Sumatra":          dict(minlat=-5.5,  maxlat=-2.0,  minlon=100.5, maxlon=104.5),
    "Kutch_India":      dict(minlat=21.5,  maxlat=24.5,  minlon=68.5,  maxlon=72.0),
    "Sichuan_China":    dict(minlat=29.5,  maxlat=32.5,  minlon=102.0, maxlon=105.5),
    "W_Australia":      dict(minlat=-32.0, maxlat=-29.0, minlon=117.0, maxlon=120.5),
    "S_Norway":         dict(minlat=58.5,  maxlat=61.5,  minlon=5.0,   maxlon=9.0),
    "Ordos_China":      dict(minlat=37.0,  maxlat=40.0,  minlon=107.5, maxlon=111.0),
}

# Catalog name mapping
CAT_NAMES = {
    "Kanto_Japan":     "Kanto_Japan",
    "Tohoku_Japan":    "Tohoku_Japan",
    "Central_Chile":   "Central_Chile",
    "Central_Turkey":  "Central_Turkey",
    "Central_Nepal":   "Central_Nepal",
    "North_Island_NZ": "North_Island_NZ",
    "Sumatra":         "Southern_Sumatra_Indonesia",
    "Kutch_India":     "Kuch_India",
    "Sichuan_China":   "Sichuan_China",
    "W_Australia":     "Western_Australia",
    "S_Norway":        "Southern_Norway",
    "Ordos_China":     "Ordos_China",
}

RES = 0.1  # grid resolution degrees

# Time axis — monthly steps 2000-01 to 2024-12
TIME_START = datetime(2000, 1, 1)
TIME_END   = datetime(2024, 12, 31)


In [4]:
def month_range(start, end):
    months = []
    current = start
    while current <= end:
        months.append(current)
        # Advance one month
        if current.month == 12:
            current = datetime(current.year + 1, 1, 1)
        else:
            current = datetime(current.year, current.month + 1, 1)
    return months

TIME_STEPS = month_range(TIME_START, TIME_END)
N_STEPS    = len(TIME_STEPS)
print(f"Time steps: {N_STEPS} months ({TIME_STEPS[0].strftime('%Y-%m')} "
      f"to {TIME_STEPS[-1].strftime('%Y-%m')})")

# Target grid
RADII_KM   = [10, 30, 50, 70, 100]
MW_THRESHS = [3.0, 3.5, 4.0, 4.5]
print(f"Target definitions: {len(RADII_KM) * len(MW_THRESHS)} "
      f"({RADII_KM} km × Mw {MW_THRESHS})")

# Temporal feature names
TEMPORAL_FEATURE_NAMES = [
    'count_30d',    'count_90d',    'count_180d',   'count_365d',
    'count_m4_180d','mean_mag_90d', 'max_mag_90d',  'max_mag_365d',
    'mean_iet_90d', 'cv_iet_90d',
    'time_since_m3','time_since_m4',
    'bvalue_180d',
    'omori_K',      'omori_p',
    'neighbour_count_30d', 'neighbour_max_mag_90d',
]
N_TEMP_FEATURES = len(TEMPORAL_FEATURE_NAMES)
print(f"Temporal features: {N_TEMP_FEATURES} — {TEMPORAL_FEATURE_NAMES}")

# ── Helper functions ──────────────────────────────────────────────────────────

def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorised haversine — lat2/lon2 can be arrays."""
    R = 6371.0
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def estimate_bvalue(mags, mc=3.0, bin_size=0.1):
    above = mags[mags >= mc]
    if len(above) < 20:
        return np.nan
    return np.log10(np.e) / (np.mean(above) - mc + bin_size/2)

def fit_omori(times_days, min_events=10):
    """
    Fit Omori-Utsu: n(t) = K / (t + c)^p
    Returns (K, p) or (nan, nan) if insufficient data.
    Uses log-linear approximation for speed.
    """
    if len(times_days) < min_events:
        return np.nan, np.nan
    t = np.sort(times_days) + 1.0  # avoid log(0)
    n = np.arange(len(t), 0, -1, dtype=float)  # reverse cumulative
    try:
        log_t = np.log(t)
        log_n = np.log(n)
        # Linear regression in log-log space
        coeffs = np.polyfit(log_t, log_n, 1)
        p = -coeffs[0]
        K = np.exp(coeffs[1])
        if 0.5 <= p <= 2.5 and K > 0:
            return round(float(K), 4), round(float(p), 4)
        return np.nan, np.nan
    except Exception:
        return np.nan, np.nan


Time steps: 300 months (2000-01 to 2024-12)
Target definitions: 20 ([10, 30, 50, 70, 100] km × Mw [3.0, 3.5, 4.0, 4.5])
Temporal features: 17 — ['count_30d', 'count_90d', 'count_180d', 'count_365d', 'count_m4_180d', 'mean_mag_90d', 'max_mag_90d', 'max_mag_365d', 'mean_iet_90d', 'cv_iet_90d', 'time_since_m3', 'time_since_m4', 'bvalue_180d', 'omori_K', 'omori_p', 'neighbour_count_30d', 'neighbour_max_mag_90d']


In [5]:
# ── Main processing loop ──────────────────────────────────────────────────────

for patch_name, bounds in PATCHES.items():
    print(f"\n{'='*60}")
    print(f"Processing {patch_name}...")

    # Load grid
    lons = np.load(feature_dir / f"{patch_name}_lons.npy")
    lats = np.load(feature_dir / f"{patch_name}_lats.npy")
    feat = np.load(feature_dir / f"{patch_name}_features.npy")
    n_lat, n_lon, _ = feat.shape
    n_cells = n_lat * n_lon

    # Grid cell centres
    grid_lats, grid_lons = np.meshgrid(lats, lons, indexing='ij')
    cell_lats = grid_lats.ravel()
    cell_lons = grid_lons.ravel()

    # Valid cells (no NaN in features)
    import json
    with open(feature_dir / "feature_names.json") as f:
        feature_names = json.load(f)
    valid_mask = ~np.any(np.isnan(feat.reshape(-1, 12)), axis=1)
    print(f"  Grid: {n_lat}×{n_lon} = {n_cells} cells "
          f"({valid_mask.sum()} valid)")

    # Load catalog
    cat_name = CAT_NAMES[patch_name]
    cat_path = cat_dir / f"{cat_name}_full_mw.csv"
    if not cat_path.exists():
        print(f"  WARNING: catalog not found — {cat_path}")
        continue

    df = pd.read_csv(cat_path)
    df['time'] = pd.to_datetime(df['time'], utc=True, format='mixed')
    df['time_naive'] = df['time'].dt.tz_localize(None)
    df = df.sort_values('time_naive').reset_index(drop=True)

    ev_lats = df['latitude'].values
    ev_lons = df['longitude'].values
    ev_mags = df['mw'].values
    ev_times = df['time_naive'].values  # numpy datetime64

    print(f"  Catalog: {len(df)} events  "
          f"({df['time_naive'].min().strftime('%Y-%m')} to "
          f"{df['time_naive'].max().strftime('%Y-%m')})")

    # ── Temporal feature tensor ───────────────────────────────────────
    # Shape: (n_lat, n_lon, N_STEPS, N_TEMP_FEATURES)
    temp_tensor = np.full(
        (n_lat, n_lon, N_STEPS, N_TEMP_FEATURES), np.nan
    )

    # ── Target tensors ────────────────────────────────────────────────
    # One per (radius, mw_thresh) combination
    # Shape per target: (n_lat, n_lon, N_STEPS)
    targets = {
        (r, mw): np.zeros((n_lat, n_lon, N_STEPS), dtype=np.int8)
        for r in RADII_KM for mw in MW_THRESHS
    }

    # Pre-compute distances: (n_events, n_cells)
    # For large patches this can be memory-intensive — process in chunks
    CHUNK = 200  # process cells in chunks

    for cell_chunk_start in range(0, n_cells, CHUNK):
        cell_chunk_end = min(cell_chunk_start + CHUNK, n_cells)
        chunk_idx = np.arange(cell_chunk_start, cell_chunk_end)

        c_lats = cell_lats[chunk_idx]
        c_lons = cell_lons[chunk_idx]

        # Distance from each event to each cell in chunk
        # Shape: (n_events, chunk_size)
        dist = np.array([
            haversine_km(ev_lats, ev_lons, c_lat, c_lon)
            for c_lat, c_lon in zip(c_lats, c_lons)
        ]).T  # (n_events, chunk_size)

        # Convert chunk flat indices to (i, j)
        chunk_ij = [
            (idx // n_lon, idx % n_lon) for idx in chunk_idx
        ]

        for t_idx, t_start in enumerate(TIME_STEPS):
            # Time windows (in days before t_start)
            t_start_np = np.datetime64(t_start)

            # Next month start for target window
            if t_start.month == 12:
                t_next = np.datetime64(
                    datetime(t_start.year + 1, 1, 1))
            else:
                t_next = np.datetime64(
                    datetime(t_start.year, t_start.month + 1, 1))

            # Days from each event to t_start (negative = before)
            days_to_t = (
                (t_start_np - ev_times).astype('timedelta64[D]')
                .astype(float)
            )

            # Boolean masks for trailing windows
            m30  = (days_to_t >= 0)  & (days_to_t <  30)
            m90  = (days_to_t >= 0)  & (days_to_t <  90)
            m180 = (days_to_t >= 0)  & (days_to_t < 180)
            m365 = (days_to_t >= 0)  & (days_to_t < 365)

            # Forward window for target
            days_to_next = (
                (ev_times - t_start_np).astype('timedelta64[D]')
                .astype(float)
            )
            m_target = (days_to_next >= 0) & (days_to_next < 30)

            for ci, (i, j) in enumerate(chunk_ij):
                if not valid_mask[chunk_idx[ci]]:
                    continue

                d = dist[:, ci]  # distances to this cell

                # ── Temporal features ─────────────────────────────
                feats = np.full(N_TEMP_FEATURES, np.nan)

                # Count features
                for wi, (mask, radius) in enumerate([
                    (m30,  50), (m90,  50),
                    (m180, 50), (m365, 50)
                ]):
                    nearby = mask & (d <= radius)
                    feats[wi] = nearby.sum()

                # Count Mw≥4 in 180d within 50km
                feats[4] = (m180 & (d <= 50) & (ev_mags >= 4.0)).sum()

                # Magnitude features (90d, 50km)
                nearby_90 = m90 & (d <= 50)
                if nearby_90.sum() > 0:
                    feats[5] = ev_mags[nearby_90].mean()
                    feats[6] = ev_mags[nearby_90].max()
                nearby_365 = m365 & (d <= 50)
                if nearby_365.sum() > 0:
                    feats[7] = ev_mags[nearby_365].max()

                # Inter-event time (90d, 50km)
                nearby_90_times = np.sort(
                    days_to_t[nearby_90])
                if len(nearby_90_times) >= 2:
                    iets = np.diff(nearby_90_times)
                    feats[8] = iets.mean()
                    feats[9] = (iets.std() / (iets.mean() + 1e-8))

                # Time since last Mw≥3 and Mw≥4 (50km)
                past = (days_to_t >= 0) & (d <= 50)
                past_m3 = past & (ev_mags >= 3.0)
                past_m4 = past & (ev_mags >= 4.0)
                if past_m3.sum() > 0:
                    feats[10] = days_to_t[past_m3].min()
                else:
                    feats[10] = 365.0
                if past_m4.sum() > 0:
                    feats[11] = days_to_t[past_m4].min()
                else:
                    feats[11] = 365.0

                # b-value (180d, 50km)
                nearby_180_mags = ev_mags[m180 & (d <= 50)]
                feats[12] = estimate_bvalue(nearby_180_mags)

                # Omori decay (50km, last Mw≥4 mainshock)
                ms_mask = past & (ev_mags >= 4.0)
                if ms_mask.sum() >= 2:
                    ms_times = days_to_t[ms_mask]
                    K, p = fit_omori(ms_times)
                    feats[13], feats[14] = K, p

                # Neighbour features (computed later in graph step)
                # Placeholder NaN for now
                feats[15] = np.nan
                feats[16] = np.nan

                temp_tensor[i, j, t_idx, :] = feats

                # ── Target computation ────────────────────────────
                for r_km in RADII_KM:
                    nearby_future = m_target & (d <= r_km)
                    for mw_thresh in MW_THRESHS:
                        hit = (nearby_future &
                               (ev_mags >= mw_thresh)).any()
                        targets[(r_km, mw_thresh)][i, j, t_idx] = \
                            int(hit)

        print(f"  t={t_idx+1}/{N_STEPS}", end='\r')

    print(f"\n  Temporal tensor shape: {temp_tensor.shape}")

    # ── Save ─────────────────────────────────────────────────────────
    np.save(
        temporal_dir / f"{patch_name}_temporal.npy",
        temp_tensor
    )

    # Save targets
    for (r_km, mw_thresh), target_arr in targets.items():
        fname = (target_dir /
                 f"{patch_name}_target_r{r_km}_mw{mw_thresh}.npy")
        np.save(fname, target_arr)

    # Quick class balance report for primary target (50km, Mw≥3.0)
    primary = targets[(50, 3.0)]
    valid_targets = primary[
        np.stack([valid_mask.reshape(n_lat, n_lon)] * N_STEPS,
                 axis=-1)
    ]
    pos_rate = valid_targets.mean()
    print(f"  Primary target (50km, Mw≥3.0): "
          f"{pos_rate*100:.1f}% positive")

    # Class balance across all target definitions
    print(f"  Class balance across all targets:")
    print(f"  {'':>8}", end='')
    for mw in MW_THRESHS:
        print(f"  Mw≥{mw}", end='')
    print()
    for r in RADII_KM:
        print(f"  r={r:>3}km ", end='')
        for mw in MW_THRESHS:
            t_arr = targets[(r, mw)]
            rate  = t_arr[
                np.stack([valid_mask.reshape(n_lat, n_lon)] * N_STEPS,
                         axis=-1)
            ].mean()
            print(f"  {rate*100:>5.1f}%", end='')
        print()

print("\n\nAll temporal features and targets saved.")
print(f"Temporal tensors: data/temporal_features/")
print(f"Target tensors:   data/targets/")


Processing Kanto_Japan...
  Grid: 27×30 = 810 cells (596 valid)
  Catalog: 2061 events  (2005-05 to 2026-05)
  t=300/300
  Temporal tensor shape: (27, 30, 300, 17)
  Primary target (50km, Mw≥3.0): 21.3% positive
  Class balance across all targets:
            Mw≥3.0  Mw≥3.5  Mw≥4.0  Mw≥4.5
  r= 10km     1.4%    1.4%    1.4%    1.4%
  r= 30km     9.7%    9.7%    9.7%    9.5%
  r= 50km    21.3%   21.3%   21.3%   20.8%
  r= 70km    32.6%   32.6%   32.6%   32.0%
  r=100km    47.5%   47.5%   47.5%   46.8%

Processing Tohoku_Japan...
  Grid: 30×30 = 900 cells (502 valid)
  Catalog: 2689 events  (2005-05 to 2026-05)
  t=300/300
  Temporal tensor shape: (30, 30, 300, 17)
  Primary target (50km, Mw≥3.0): 21.1% positive
  Class balance across all targets:
            Mw≥3.0  Mw≥3.5  Mw≥4.0  Mw≥4.5
  r= 10km     1.7%    1.7%    1.7%    1.7%
  r= 30km    10.2%   10.2%   10.2%   10.0%
  r= 50km    21.1%   21.1%   21.1%   20.7%
  r= 70km    32.5%   32.5%   32.5%   31.9%
  r=100km    48.1%   48.1%  

## Insights

### Overview

<p>Temporal feature engineering converts the processed earthquake catalogs into structured time-series representations suitable for graph neural network training. For each patch, a four-dimensional tensor of shape (n_lat, n_lon, n_timesteps, n_temporal_features) was computed, where each cell at each monthly time step receives a vector of 17 temporal features derived exclusively from the catalog history preceding that time step. Simultaneously, binary target tensors were computed for all 20 combinations of spatial radius (10, 30, 50, 70, 100 km) and magnitude threshold (Mw ≥ 3.0, 3.5, 4.0, 4.5), producing a comprehensive grid of prediction objectives that enables systematic evaluation of model performance across different spatial scales and event sizes.</p>

### Time-axis

<p>A monthly time step was adopted as the primary temporal resolution, producing 300 time steps spanning January 2000 to December 2024. Monthly resolution represents a deliberate balance between temporal granularity and statistical stability — daily resolution would produce mostly empty windows for sparse patches, while quarterly resolution would obscure the rapid rate changes following large mainshocks that are among the most informative temporal signals. All temporal features are computed as trailing windows ending at the time step start, ensuring strict temporal separation between features and target: features at time t use only catalog data from the period [t-365, t-1], and the target at time t uses events from [t, t+30]. No future data ever enters any feature at any time step.</p>

### Temporal Features

<p>Seventeen temporal features were computed per cell per time step across four categories. Count features capture seismicity rate at multiple timescales: event counts in 30, 90, 180, and 365-day trailing windows within 50 km of the cell centroid, plus a count of Mw ≥ 4.0 events in the 180-day window. These multi-scale counts allow the model to distinguish short-term aftershock bursts from sustained elevated seismicity. Magnitude features capture the size distribution of recent activity: mean and maximum magnitude in the 90-day window, and maximum magnitude in the 365-day window. The 365-day maximum is particularly important as a proxy for recent large events that may have reset the local stress state. Temporal spacing features characterise the regularity of recent seismicity: mean inter-event time and coefficient of variation of inter-event times in the 90-day window, plus time since last Mw ≥ 3.0 and Mw ≥ 4.0 event within 50 km. The coefficient of variation of inter-event times is a physically motivated feature — changes in seismicity regularity have been proposed as precursory signals in the seismological literature and provide information complementary to simple rate counts. Statistical physics features encode the scaling properties of recent seismicity: b-value estimated from events in the 180-day window (set to NaN if fewer than 20 events are available), and Omori-Utsu decay parameters K and p fitted to the aftershock sequence following the most recent Mw ≥ 4.0 event within 50 km. The Omori parameters capture aftershock productivity (K) and decay rate (p), both of which vary systematically with tectonic regime and fault geometry. Two neighbour features (spatial count and maximum magnitude aggregated from adjacent cells) are reserved as NaN placeholders to be filled during graph construction, when adjacency information becomes available.</p>

### Target Variable Design

<p>The 20-target grid was designed to systematically probe model performance across spatial scale and event size simultaneously. The primary operational target is defined as Mw ≥ 3.0 within 50 km in the next 30 days — a definition broad enough to produce meaningful event rates in active patches while remaining spatially specific enough to be useful for hazard communication. The grid extension to smaller radii (10, 30 km) tests whether the model can localise hazard at fine spatial scales, while larger radii (70, 100 km) test regional-scale predictability. The magnitude dimension tests whether the model captures the full magnitude-frequency distribution or only responds to large events.</p>

### Class Balance Insight

<p>The class balance analysis across all 20 target definitions reveals a critical finding that shapes the entire modelling strategy: for most patches, the magnitude threshold dimension of the target grid collapses to effective irrelevance above the patch Mc. Kanto, with Mc = 4.4, shows nearly identical positive rates across all four magnitude thresholds at every spatial radius — 21.3% at Mw ≥ 3.0 and 20.8% at Mw ≥ 4.5 at 50 km — because the catalog contains essentially no events between Mw 3.0 and 4.4 after Mc trimming. The same pattern holds for Tohoku (Mc = 4.4), Turkey (Mc = 4.3), and Sichuan (Mc = 4.3). The Mw threshold variation produces meaningful class balance differences only for patches with lower Mc: Chile (Mc = 4.0) drops from 13.2% to 11.5% across the magnitude range at 50 km, and NZ (Mc = 4.0) drops from 13.3% to 10.8%. This finding has a direct implication for the experimental design — for high-Mc patches the effective target grid is five-dimensional (radius only), and the 20-target framing should be interpreted accordingly in the Results section.</p>

<p>Primary target class balance at 50 km / Mw ≥ 3.0 varies substantially across patches, spanning five orders of magnitude from 21.3% (Kanto) to 0.0% (Norway). This range reflects both genuine differences in seismicity rates across tectonic regimes and the catalog sparsity of low-seismicity patches. The active subduction patches (Kanto 21.3%, Tohoku 21.1%) show the most balanced class distributions and will contribute the strongest training signal. Chile (13.2%) and NZ (13.3%) are moderately imbalanced but workable. Turkey (3.1%), Sichuan (3.6%), and Nepal (1.8%) show significant imbalance requiring focal loss or class weighting. Kutch (0.6%), Ordos (0.6%), Australia (0.3%), and Norway (0.0%) have severe imbalance — these patches serve as transfer targets and false-positive stress tests rather than primary training contributors.</p>

<p>Norway presents a special case: zero positive labels across all 20 target definitions. With only 4 catalog events over 24 years, none of which produce a 30-day forward window with a qualifying event, the Norway patch cannot contribute to AUC computation at any threshold. Its role in the evaluation framework is therefore purely diagnostic — a model that assigns elevated hazard probabilities to Norwegian cells at any time step is learning spurious patterns, and the mean predicted probability for Norway serves as a false-positive rate indicator rather than a classification metric. Norway will be excluded from AUC-based evaluation but included in calibration analysis.</p>

### Implications for Model Training

<p>The class imbalance landscape directly motivates the use of focal loss with γ=2 as the training objective rather than standard binary cross-entropy. Focal loss down-weights the contribution of easy negative examples — the overwhelming majority of cell-time-step combinations in sparse patches — allowing the model to focus on the minority positive class and the genuinely ambiguous boundary cases. Per-patch class weights will additionally be computed from the training period positive rates and applied as loss multipliers to prevent dense patches (Kanto, Tohoku) from dominating the gradient signal at the expense of sparse but geologically important patches (Nepal, Kutch).</p>

<p>For the MCPC evaluation, AUC-PR (precision-recall curve area) will be reported alongside AUC-ROC as the primary metric for patches with positive rates below 5%. At very low positive rates AUC-ROC is dominated by true negative performance and produces misleadingly high scores — a model that predicts all negatives achieves AUC-ROC ≈ 0.99 on Norway but AUC-PR ≈ 0.0. AUC-PR correctly penalises this behaviour and is the appropriate metric for rare event prediction.</p>